# VAE 作业二：将 FengHZ 的 VAE 训练技巧融入八维卷积 VAE

**实验目标**：在已有 `conv_mse_z8` 的基础上，以相同的卷积网络、数据划分和重建 MSE 尺度，比较固定 KL、单次线性预热、周期 KL 退火；检查数字 4/7/8 的重建结构、先验生成质量和潜变量利用。

**资料来源**：
- FengHZ (2018), *Some useful tricks in training variational autoencoder*: https://fenghz.github.io/VAE-train-tips/
- Fu et al. (2019), *Cyclical Annealing Schedule*: https://github.com/haofuml/cyclical_annealing

来自原文章并落实到代码：MSE/KL 的损失缩放审计、预测 log-variance、避免 NaN、适度梯度裁剪、独立绘制 MSE 和 KL、逐维 KL 和 PCA 潜变量诊断。
周期退火来自上述额外论文，不是 FengHZ 博客自行提出的结论。

**最重要的尺度说明**：原 `mse.mean() + 0.0005*KL` 不能直接改为 `mse.sum()/batch + 0.0005*KL`：后者将重建项相对放大 784 倍。本 Notebook 保持与之前相同的每像素 MSE 和每样本逐维 KL 求和。

## Kaggle 使用
1. Add Input 挂载此前的 `CNNmodel_database_MNIST`；默认读取原 IDX 路径，不依赖 torchvision 下载。
2. 开启 GPU，`FAST_DEV_RUN=False` 后 Run All；本实验无需上传旧 `.pt`，所有候选均可从头训练。
3. 需要复用以前 `conv_mse_z8` 的 *已核验* 40轮权重时，可启用 `REUSE_OLD_Z8_BASELINE=True` 并挂载对应 Kaggle Dataset；复用模型的旧训练算法可能与本次新增梯度裁剪略有不同，须在报告说明。
4. 选模只用验证集及辅助分类器的先验生成代理指标，正式测试集仅在 `FINAL_TEST_CONFIRMED=True` 后使用。
5. 最终自动压缩为 `/kaggle/working/VAE_8D_Training_Tips_Results.zip`。

In [1]:
from pathlib import Path
import os, math, csv, random, struct, json, zipfile, shutil, copy, gc, hashlib
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt

# ======= 需要修改时，从这里开始 =======
FAST_DEV_RUN = os.getenv('VAE_FAST_DEV_RUN','0') == '1'
DATA_ROOT = Path(os.getenv('VAE_IDX_ROOT', '/kaggle/input/datasets/yuesun0502/cnnmodel-database-minist'))
WORK = Path(os.getenv('VAE_OUTPUT_ROOT', '/kaggle/working' if Path('/kaggle/working').is_dir() else str(Path.cwd())))
OUTPUT_ROOT = WORK / 'hw2_results' / ('vae_training_tips_smoke' if FAST_DEV_RUN else 'vae_training_tips_z8_v1')
MODEL_DIR = OUTPUT_ROOT/'models'; TABLE_DIR = OUTPUT_ROOT/'tables'; FIG_DIR=OUTPUT_ROOT/'figures'
for p in (MODEL_DIR, TABLE_DIR, FIG_DIR):p.mkdir(parents=True,exist_ok=True)

SEED=42
SPLIT_SEED=2026
LATENT_DIM=8
EPOCHS=40
BATCH_SIZE=256
LR=1e-4
BETA_MAX=0.0005
GRAD_NORM_LIMIT=5.0  # 适中范数裁剪，而非复制博客中 1e5 的极大数值
LOGVAR_MIN, LOGVAR_MAX = -12.0, 8.0
VAL_COUNT=5000
SAMPLE_REPEATS=3
CLASSIFIER_EPOCHS=3
PRIOR_EVAL_N=2000
PRIOR_SHOW_N=64
WARMUP_EPOCHS=10
CYCLE_COUNT=4
CYCLE_RAMP_FRACTION=0.5
RUN_FREE_BITS_EXPERIMENT=False  # 可选扩展；与文章内容区分，需额外训练 40 轮
FREE_BITS_PER_DIM=0.05         # 不是强制KL>0.05，而是低于阈值时停止施加额外KL梯度
REUSE_OLD_Z8_BASELINE=False   # 默认全部新训，保证梯度裁剪等训练设置严格一致
EXTRA_OLD_CHECKPOINT_PATHS=[]
FINAL_TEST_CONFIRMED=False
FINAL_RUN_NAME='auto'           # 或指定: constant / linear_warmup / cyclical / cyclical_free_bits

if FAST_DEV_RUN:
    EPOCHS=4; BATCH_SIZE=16; VAL_COUNT=16; CLASSIFIER_EPOCHS=1
    PRIOR_EVAL_N=24; PRIOR_SHOW_N=16; SAMPLE_REPEATS=1
    WARMUP_EPOCHS=2; CYCLE_COUNT=2
    REUSE_OLD_Z8_BASELINE=False
    torch.set_num_threads(min(2,torch.get_num_threads()))

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
seed_all(SEED)
print('device:',device, 'root:',OUTPUT_ROOT, 'FAST_DEV_RUN:',FAST_DEV_RUN)

device: cuda root: /kaggle/working/hw2_results/vae_training_tips_z8_v1 FAST_DEV_RUN: False


## 1. MNIST IDX 数据与可复现划分
与上一次三阶段实验一致：使用 seed 2026 从原 60000 训练样本划分出 5000 验证样本；不使用最终测试集进行调参。

In [2]:
IDX={
 'train_images':DATA_ROOT/'train-images-idx3-ubyte'/'train-images.idx3-ubyte',
 'train_labels':DATA_ROOT/'train-labels-idx1-ubyte'/'train-labels.idx1-ubyte',
 'test_images': DATA_ROOT/'t10k-images-idx3-ubyte'/'t10k-images.idx3-ubyte',
 'test_labels': DATA_ROOT/'t10k-labels-idx1-ubyte'/'t10k-labels.idx1-ubyte',
}
missing=[str(p) for p in IDX.values() if not p.exists()]
if missing:raise FileNotFoundError('找不到MNIST IDX文件：\n'+'\n'.join(missing)+'\n请先 Kaggle > Add Input 挂载原数据集。')

def read_images(path):
    with path.open('rb') as f:
        head=f.read(16)
        if len(head)!=16:raise ValueError('IDX header truncated')
        magic,n,h,w=struct.unpack('>IIII',head)
        arr=np.frombuffer(f.read(),dtype=np.uint8).copy()
    if (magic,h,w)!=(2051,28,28) or len(arr)!=n*h*w:raise ValueError('Bad image IDX '+str(path))
    return torch.from_numpy(arr.reshape(n,1,h,w))

def read_labels(path):
    with path.open('rb') as f:
        magic,n=struct.unpack('>II',f.read(8));arr=np.frombuffer(f.read(),dtype=np.uint8).copy()
    if magic!=2049 or len(arr)!=n:raise ValueError('Bad label IDX '+str(path))
    return torch.from_numpy(arr).long()

class MNISTIDX(Dataset):
    def __init__(self,part,limit=None):
        self.x=read_images(IDX[f'{part}_images']);self.y=read_labels(IDX[f'{part}_labels'])
        assert len(self.x)==len(self.y)
        if limit:self.x=self.x[:limit];self.y=self.y[:limit]
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return self.x[i].float()/255.,self.y[i]

train_full=MNISTIDX('train',limit=96 if FAST_DEV_RUN else None)
test_data=MNISTIDX('test',limit=40 if FAST_DEV_RUN else None)
val_n=min(VAL_COUNT,len(train_full)//4)
perm=torch.randperm(len(train_full),generator=torch.Generator().manual_seed(SPLIT_SEED))
train_idx=perm[:-val_n].tolist();val_idx=perm[-val_n:].tolist()
train_set=Subset(train_full,train_idx);val_set=Subset(train_full,val_idx)
val_loader=DataLoader(val_set,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
test_loader=DataLoader(test_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
np.savez_compressed(TABLE_DIR/'split_indices.npz',train=np.asarray(train_idx),val=np.asarray(val_idx))
def train_loader_for(epoch,seed=SEED):
    g=torch.Generator().manual_seed(seed*10000+epoch)
    return DataLoader(train_set,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,generator=g)
print('train:',len(train_set),'val:',len(val_set),'test held out:',len(test_data))

train: 55000 val: 5000 test held out: 10000


## 2. 保持已有八维卷积 VAE 架构；使用数值稳定的 logvar 参数化
网络层形状与此前 `conv_mse_z8` 完全一致。`torch.exp(0.5*logvar)` 由对数方差计算标准差；在计算指数前裁剪 logvar 以降低溢出风险。

In [3]:
class ConvVAE(nn.Module):
    def __init__(self,d=8):
        super().__init__();self.d=d
        self.features=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),nn.ReLU(),nn.Conv2d(32,64,4,2,1),nn.ReLU(),
            nn.Flatten(),nn.Linear(64*7*7,128),nn.ReLU())
        self.mu=nn.Linear(128,d);self.lv=nn.Linear(128,d)
        self.up=nn.Sequential(nn.Linear(d,128),nn.ReLU(),nn.Linear(128,64*7*7),nn.ReLU())
        self.deconv=nn.Sequential(nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),nn.ConvTranspose2d(32,1,4,2,1))
    def encode(self,x):
        h=self.features(x)
        return self.mu(h),self.lv(h).clamp(LOGVAR_MIN,LOGVAR_MAX)
    def decode_logits(self,z):return self.deconv(self.up(z).reshape(-1,64,7,7))
    def decode(self,z):return torch.sigmoid(self.decode_logits(z))
    def forward(self,x):
        mu,lv=self.encode(x)
        z=mu+torch.exp(0.5*lv)*torch.randn_like(mu)
        return self.decode_logits(z),mu,lv

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),
            nn.Linear(32*7*7,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x):return self.net(x)

@torch.no_grad()
def count_parameters(model):return sum(p.numel() for p in model.parameters())
print('VAE parameters:', count_parameters(ConvVAE(LATENT_DIM)))

VAE parameters: 875985


## 3. 核对 MSE 与 KL 尺度；安全实现 free bits（可选）
原文章强调每样本像素求和与每像素平均的差异，我们通过数值断言确保二者的正确换算。
对角高斯每个维度的 KL 为 `0.5*(mu**2 + exp(logvar) - 1 - logvar)`；逐维求和后再对 batch 取均值。
注意：free bits 只让低于阈值的 KL 不再受额外正则化梯度推动；不会自动把该维的真实 KL 提高到阈值以上。

In [4]:
def kl_per_item_dim(mu,lv):return 0.5*(mu.square()+lv.exp()-1-lv)

def get_losses(logits,x,mu,lv,beta,free_bits=0.0):
    xhat=torch.sigmoid(logits)
    pixel_mse=F.mse_loss(xhat,x,reduction='mean')  # 与此前作业保持一致
    sse=pixel_mse*784  # 按图像求和的SSE，只用于报告等效尺度，不在每个batch重复同步检查
    kl_dims=kl_per_item_dim(mu,lv)
    raw_kl=kl_dims.sum(1).mean()
    penalized_kl=(torch.clamp_min(kl_dims.mean(0),free_bits).sum() if free_bits>0 else raw_kl)
    loss=pixel_mse+beta*penalized_kl
    return loss,{'mse':pixel_mse,'kl_raw':raw_kl,'kl_penalty':penalized_kl,'sse':sse}

xprobe=torch.rand(4,1,28,28); modelprobe=ConvVAE(LATENT_DIM)
logits,mu,lv=modelprobe(xprobe);loss,_=get_losses(logits,xprobe,mu,lv,0.0005)
assert torch.isfinite(loss) and logits.shape==xprobe.shape
with torch.no_grad():
    mse_check=F.mse_loss(torch.sigmoid(logits),xprobe,reduction='mean')
    sum_check=F.mse_loss(torch.sigmoid(logits),xprobe,reduction='sum')/len(xprobe)
    assert torch.allclose(sum_check,784*mse_check,atol=1e-4,rtol=1e-5)
loss.backward()
del modelprobe
print('loss scaling and forward/backward smoke checks passed')

loss scaling and forward/backward smoke checks passed


## 4. 固定、单调预热、周期退火：只改变 β 调度
三组主实验的 β 最大值都为 0.0005，MSE 与 KL 尺度完全相同；周期退火采用每个周期先线性增长后固定在目标权重的形式。
**周期退火是额外文献提出的方法，不能认为必然改善 MNIST 的生成结果。**

In [5]:
def beta_value(kind,epoch,total):
    if kind=='constant':return BETA_MAX
    if kind=='linear_warmup':
        return BETA_MAX*min(1.0,epoch/max(WARMUP_EPOCHS,1))
    if kind in ('cyclical','cyclical_free_bits'):
        cycles=max(1,CYCLE_COUNT)
        # 以步数为单位循环；最后一个周期的最大β可能略受整数取整影响
        period=total/cycles
        position=((epoch-1)%period)/period
        return BETA_MAX*min(1.0, position/max(CYCLE_RAMP_FRACTION,1e-8))
    raise ValueError(kind)

EXPERIMENTS=[
    {'name':'constant','schedule':'constant','free_bits':0.0},
    {'name':'linear_warmup','schedule':'linear_warmup','free_bits':0.0},
    {'name':'cyclical','schedule':'cyclical','free_bits':0.0},
]
if RUN_FREE_BITS_EXPERIMENT:EXPERIMENTS.append({'name':'cyclical_free_bits','schedule':'cyclical_free_bits','free_bits':FREE_BITS_PER_DIM})

fig,ax=plt.subplots(figsize=(9,3.7))
for cfg in EXPERIMENTS:
    ax.plot(range(1,EPOCHS+1),[beta_value(cfg['schedule'],e,EPOCHS) for e in range(1,EPOCHS+1)],label=cfg['name'])
ax.set_xlabel('Epoch');ax.set_ylabel('KL weight beta');ax.set_title('KL schedules: same maximum beta');ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(FIG_DIR/'kl_schedules.png',dpi=180);plt.close(fig)

## 5. 保存并恢复模型/优化器/随机状态，避免此前 Kaggle CUDA RNG 类型错误
默认从零训练三组模型；可选复用 *已核对元数据* 的旧八维基准。只读取自己训练或明确上传的可信 `.pt` 文件。

In [6]:
def cpu_copy(obj):
    if torch.is_tensor(obj):return obj.detach().cpu().clone()
    if isinstance(obj,dict):return {k:cpu_copy(v) for k,v in obj.items()}
    if isinstance(obj,list):return [cpu_copy(v) for v in obj]
    if isinstance(obj,tuple):return tuple(cpu_copy(v) for v in obj)
    return obj

def csv_write(path,rows):
    rows=list(rows)
    if not rows:return
    with path.open('w',newline='',encoding='utf-8-sig') as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0]));wr.writeheader();wr.writerows(rows)

def make_signature(cfg):
    return dict(experiment=cfg['name'], schedule=cfg['schedule'],free_bits=cfg['free_bits'],
        latent_dim=LATENT_DIM, epochs=EPOCHS,beta_max=BETA_MAX,lr=LR,batch_size=BATCH_SIZE,
        grad_clip=GRAD_NORM_LIMIT,logvar_min=LOGVAR_MIN,logvar_max=LOGVAR_MAX,
        split_seed=SPLIT_SEED,train_count=len(train_set),val_count=len(val_set),seed=SEED,
        architecture='conv_32_64_128_transpose', loss='pixel_mean_mse_plus_kl',
        cycles=CYCLE_COUNT,ramp_fraction=CYCLE_RAMP_FRACTION,warmup_epochs=WARMUP_EPOCHS)

def save_checkpoint(path,model,opt,history,epoch,cfg):
    obj={'model':cpu_copy(model.state_dict()),'optimizer':cpu_copy(opt.state_dict()),
         'history':history,'epoch':epoch,'signature':make_signature(cfg),
         'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),
         'rng_torch':torch.get_rng_state().cpu(),
         'rng_cuda':[s.cpu() for s in torch.cuda.get_rng_state_all()] if torch.cuda.is_available() else []}
    tmp=path.with_suffix('.tmp');torch.save(obj,tmp);tmp.replace(path)
    if epoch in (20,EPOCHS):torch.save(obj,path.parent/f'checkpoint_epoch_{epoch}.pt')

def load_checkpoint(path,model,opt,cfg):
    obj=torch.load(path,map_location='cpu',weights_only=False) # 仅加载自己已验证来源的文件
    if obj.get('signature')!=make_signature(cfg):
        raise RuntimeError(f'检查点配置不一致: {path}，请改新输出目录，不要覆盖或混用')
    model.load_state_dict(obj['model']);opt.load_state_dict(obj['optimizer'])
    torch.set_rng_state(obj['rng_torch'].cpu().to(torch.uint8).contiguous())
    np.random.set_state(obj['rng_numpy']);random.setstate(obj['rng_python'])
    states=obj.get('rng_cuda',[])
    if device.type=='cuda' and len(states)==torch.cuda.device_count():
        torch.cuda.set_rng_state_all([s.cpu().to(torch.uint8).contiguous() for s in states])
    return int(obj['epoch']),list(obj['history'])

def try_import_legacy_z8(model,cfg):
    if not REUSE_OLD_Z8_BASELINE or FAST_DEV_RUN or cfg['name']!='constant':return None
    candidates=[Path(p) for p in EXTRA_OLD_CHECKPOINT_PATHS]
    ki=Path('/kaggle/input')
    if ki.exists():candidates+=list(ki.rglob('checkpoint_epoch_40*.pt'))
    for p in candidates:
        if not p.is_file():continue
        try:
            obj=torch.load(p,map_location='cpu',weights_only=False) # 用户明确挂载的可信旧文件
            c=obj.get('config',{})
            matched=(c.get('name')=='conv_mse_z8' and c.get('latent_dim')==8 and
                     c.get('beta')==BETA_MAX and c.get('seed')==SEED and
                     c.get('train_count')==len(train_set) and c.get('val_count')==len(val_set) and
                     c.get('split_seed')==SPLIT_SEED and c.get('batch_size')==BATCH_SIZE and
                     int(obj.get('epoch',-1))==EPOCHS and abs(c.get('lr',LR)-LR)<1e-12)
            if matched:
                model.load_state_dict(obj['model'])
                return str(p)
        except Exception as exc:print('Skip optional checkpoint',p.name,type(exc).__name__)
    return None

## 6. 实际训练与逐 epoch MSE / KL 曲线
`grad_clip_norm_` 控制整体梯度范数；如果 loss 或梯度出现 NaN/Inf，直接报出具体模型与 Epoch，不静默继续。

In [7]:
def train_or_load(cfg):
    fold=MODEL_DIR/cfg['name'];fold.mkdir(parents=True,exist_ok=True)
    seed_all(SEED)
    model=ConvVAE(LATENT_DIM).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    latest=fold/'checkpoint_latest.pt';history=[];finished=0
    if latest.exists():
        finished,history=load_checkpoint(latest,model,opt,cfg)
        if finished>EPOCHS:raise RuntimeError('Checkpoint newer than requested training target')
        print(cfg['name'],'resume from epoch',finished)
    else:
        legacy_path=try_import_legacy_z8(model,cfg)
        if legacy_path:
            finished=EPOCHS
            (fold/'legacy_import.json').write_text(json.dumps({'source':legacy_path,
                'caution':'Imported baseline trained without this notebook gradient clipping; compare with caution'},ensure_ascii=False,indent=2),encoding='utf-8')
            print('Imported validated previous z8 baseline:',legacy_path)
            save_checkpoint(latest,model,opt,history,finished,cfg)
    for epoch in range(finished+1,EPOCHS+1):
        model.train();tot_mse=tot_kl=tot_pen=0.;N=0
        beta=beta_value(cfg['schedule'],epoch,EPOCHS)
        for x,_ in train_loader_for(epoch):
            x=x.to(device);logits,mu,lv=model(x)
            loss,components=get_losses(logits,x,mu,lv,beta,cfg['free_bits'])
            if not torch.isfinite(loss):raise FloatingPointError(f'NaN/Inf loss: {cfg["name"]} epoch={epoch}')
            opt.zero_grad(set_to_none=True);loss.backward()
            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_NORM_LIMIT,error_if_nonfinite=True)
            opt.step()
            b=len(x);N+=b
            tot_mse+=b*float(components['mse'].detach().item());tot_kl+=b*float(components['kl_raw'].detach().item())
            tot_pen+=b*float(components['kl_penalty'].detach().item())
        h={'epoch':epoch,'beta':beta,'train_mse':tot_mse/N,'train_kl_raw':tot_kl/N,
            'train_kl_penalty':tot_pen/N,'train_loss':tot_mse/N+beta*tot_pen/N}
        history.append(h)
        if epoch==1 or epoch%5==0 or epoch==EPOCHS:
            print(cfg['name'],f'epoch {epoch}/{EPOCHS}',f'MSE={h["train_mse"]:.5f}',f'KL={h["train_kl_raw"]:.3f}',f'beta={beta:.6f}')
        if epoch%5==0 or epoch==EPOCHS:save_checkpoint(latest,model,opt,history,epoch,cfg)
    csv_write(fold/'history.csv',history)
    model.eval();return model,history

models={};histories={}
for cfg in EXPERIMENTS:
    m,h=train_or_load(cfg)
    models[cfg['name']]=m;histories[cfg['name']]=h
    # 逐轮训练后将模型转到 CPU，以免多个模型同时占满 GPU；评估时再移回 GPU。
    m.cpu()
    if torch.cuda.is_available():torch.cuda.empty_cache()

fig,axes=plt.subplots(1,3,figsize=(16,4))
for name,h in histories.items():
    if not h:continue
    axes[0].plot([v['epoch'] for v in h],[v['train_mse'] for v in h],label=name)
    axes[1].plot([v['epoch'] for v in h],[v['train_kl_raw'] for v in h],label=name)
    axes[2].plot([v['epoch'] for v in h],[v['train_loss'] for v in h],label=name)
for ax,title in zip(axes,['Training pixel MSE','Training raw total KL','Training weighted total loss (different beta schedules)']):
    ax.set_title(title);ax.set_xlabel('Epoch');ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(FIG_DIR/'training_curves.png',dpi=180);plt.close(fig)

constant epoch 1/40 MSE=0.10915 KL=11.223 beta=0.000500
constant epoch 5/40 MSE=0.04123 KL=9.671 beta=0.000500
constant epoch 10/40 MSE=0.02577 KL=13.704 beta=0.000500
constant epoch 15/40 MSE=0.02325 KL=14.344 beta=0.000500
constant epoch 20/40 MSE=0.02183 KL=14.722 beta=0.000500
constant epoch 25/40 MSE=0.02086 KL=15.018 beta=0.000500
constant epoch 30/40 MSE=0.02016 KL=15.195 beta=0.000500
constant epoch 35/40 MSE=0.01960 KL=15.347 beta=0.000500
constant epoch 40/40 MSE=0.01918 KL=15.468 beta=0.000500
linear_warmup epoch 1/40 MSE=0.10606 KL=44.057 beta=0.000050
linear_warmup epoch 5/40 MSE=0.04737 KL=11.520 beta=0.000250
linear_warmup epoch 10/40 MSE=0.02569 KL=13.950 beta=0.000500
linear_warmup epoch 15/40 MSE=0.02268 KL=14.639 beta=0.000500
linear_warmup epoch 20/40 MSE=0.02111 KL=15.014 beta=0.000500
linear_warmup epoch 25/40 MSE=0.02015 KL=15.248 beta=0.000500
linear_warmup epoch 30/40 MSE=0.01947 KL=15.405 beta=0.000500
linear_warmup epoch 35/40 MSE=0.01894 KL=15.533 beta=0.000

## 7. 冻结分类器：先检验对真实 MNIST 验证集的准确率
VAE 仍是无监督训练；分类器只用于验证重建类别一致率和先验生成代理指标。低于 95% 时不使用这些分类器指标选模。

In [8]:
def reference_classifier_paths():
    k=Path('/kaggle/input');ret=[]
    if k.exists():
        ret+=list(k.rglob('mnist_reference_classifier.pt'))
        ret+=list(k.rglob('reference_classifier.pt'))
    ret+=[OUTPUT_ROOT/'reference_classifier.pt']
    return list(dict.fromkeys(p for p in ret if p.is_file()))

def get_classifier():
    seed_all(987);clf=DigitClassifier().to(device);loaded=False
    for p in reference_classifier_paths():
        try:
            obj=torch.load(p,map_location='cpu',weights_only=True)
            state=obj.get('state_dict',obj) if isinstance(obj,dict) else obj
            clf.load_state_dict(state)
            loaded=True;print('Loaded classifier:',p);break
        except Exception as e:print('Skip incompatible classifier:',p.name,type(e).__name__)
    if not loaded:
        optimizer=torch.optim.Adam(clf.parameters(),lr=1e-3)
        for ep in range(1,CLASSIFIER_EPOCHS+1):
            clf.train();hits=N=0
            for x,y in train_loader_for(ep,987):
                x,y=x.to(device),y.to(device)
                optloss=F.cross_entropy(clf(x),y)
                optimizer.zero_grad(set_to_none=True);optloss.backward();optimizer.step()
                hits+=int((clf(x).argmax(1)==y).sum());N+=len(y)
            print('Classifier epoch',ep,'training agreement',round(hits/N,4))
        torch.save(cpu_copy(clf.state_dict()),OUTPUT_ROOT/'reference_classifier.pt')
    clf.eval();hits=N=0
    with torch.no_grad():
        for x,y in val_loader:
            pred=clf(x.to(device)).argmax(1).cpu();hits+=int((pred==y).sum());N+=len(y)
    accuracy=hits/N
    print('REAL validation classifier accuracy=',round(accuracy,4),'valid proxy=',accuracy>=.95)
    (TABLE_DIR/'classifier_validation.json').write_text(json.dumps({'validation_accuracy':accuracy,'proxy_valid':accuracy>=.95},indent=2),encoding='utf-8')
    for p in clf.parameters():p.requires_grad_(False)
    return clf,accuracy
classifier,classifier_accuracy=get_classifier()
PROXY_VALID=classifier_accuracy>=0.95

Classifier epoch 1 training agreement 0.8693
Classifier epoch 2 training agreement 0.9649
Classifier epoch 3 training agreement 0.9781
REAL validation classifier accuracy= 0.9754 valid proxy= True


## 8. 统一验证集指标（测试集不参与选模）、重建类别混淆、先验生成质量
准确率仅是冻结分类器的代理评价；分类器对原图误判的样本另行排除后评价，避免将分类器自身错误算在VAE头上。

In [9]:
@torch.no_grad()
def evaluate_recon(model,loader,repeats=SAMPLE_REPEATS):
    model.eval();n=0;sse=sample_sse=kl_total=0.0
    counts=np.zeros(10,int);digit_sse=np.zeros(10,float)
    conf=np.zeros((10,10),dtype=np.int64);conf_original=np.zeros((10,10),dtype=np.int64)
    conf_on_original_correct=np.zeros((10,10),dtype=np.int64);num_original_correct=np.zeros(10,dtype=np.int64)
    per_dim=np.zeros(model.d,float);mus=[]
    for x,y in loader:
        x=x.to(device);y=y.to(device);b=len(x)
        mu,lv=model.encode(x);rec=model.decode(mu)
        mse_item=(rec-x).square().flatten(1).mean(1)
        sse+=float(mse_item.sum());klv=kl_per_item_dim(mu,lv)
        kl_total+=float(klv.sum());per_dim+=klv.sum(0).cpu().numpy()
        n+=b;targets=y.cpu().numpy()
        counts+=np.bincount(targets,minlength=10)
        digit_sse+=np.bincount(targets,weights=mse_item.cpu().numpy(),minlength=10)
        mus.append(mu.cpu().numpy())
        for _ in range(repeats):
            z=mu+torch.exp(0.5*lv)*torch.randn_like(mu)
            sample_sse+=float((model.decode(z)-x).square().flatten(1).mean(1).sum())/repeats
        if PROXY_VALID:
            original=classifier(x).argmax(1).cpu().numpy()
            predicted=classifier(rec).argmax(1).cpu().numpy()
            np.add.at(conf,(targets,predicted),1);np.add.at(conf_original,(targets,original),1)
            good_original=(original==targets)
            np.add.at(conf_on_original_correct,(targets[good_original],predicted[good_original]),1)
            num_original_correct+=np.bincount(targets[good_original],minlength=10)
    mean=np.concatenate(mus,axis=0);centered=mean-mean.mean(0,keepdims=True)
    # PCA/SVD 对编码均值做方差解释率，作为潜变量维度诊断，而非直接裁剪已训练网络
    sv=np.linalg.svd(centered,compute_uv=False)
    eigen=sv**2; frac=(eigen/eigen.sum()) if eigen.sum()>0 else np.zeros(len(eigen))
    k90=int(np.searchsorted(np.cumsum(frac),0.90)+1) if eigen.sum()>0 else 0
    result={'mean_mse':sse/n,'sample_mse':sample_sse/n,'total_kl':kl_total/n,
        'kl_by_dim':(per_dim/n).tolist(),'pca_variance_ratio':frac.tolist(),'pca_k90':k90,
        'mean_var_by_dim':centered.var(axis=0).tolist(),
        'per_digit_mse':{str(i):float(digit_sse[i]/counts[i]) if counts[i] else None for i in range(10)},
        'n':n}
    if PROXY_VALID:
        result.update(reconstruction_class_accuracy=float(np.trace(conf)/n),
            per_digit_reconstruction_accuracy={str(i):float(conf[i,i]/counts[i]) if counts[i] else None for i in range(10)},
            confusion=conf.tolist(),original_confusion=conf_original.tolist(),
            reconstruction_accuracy_on_original_correct=float(np.trace(conf_on_original_correct)/max(1,num_original_correct.sum())),
            per_digit_accuracy_on_original_correct={str(i):float(conf_on_original_correct[i,i]/num_original_correct[i]) if num_original_correct[i] else None for i in range(10)},
            original_classifier_accuracy=float(np.trace(conf_original)/n),
            directed_confusion={'4_to_9':int(conf[4,9]),'7_to_9':int(conf[7,9]),
                                '8_to_5':int(conf[8,5]),'8_to_6':int(conf[8,6])})
    return result

@torch.no_grad()
def prior_images(model,n,seed=9881):
    model.eval();g=torch.Generator(device=device.type).manual_seed(seed)
    return torch.cat([model.decode(torch.randn(min(128,n-i),model.d,device=device,generator=g)).cpu()
         for i in range(0,n,128)])

@torch.no_grad()
def eval_prior(model,n=PRIOR_EVAL_N):
    if not PROXY_VALID:return {}
    ims=prior_images(model,n)
    counts=np.zeros(10,int);confident=0
    for i in range(0,n,128):
        prob=classifier(ims[i:i+128].to(device)).softmax(1)
        v,y=prob.max(1);confident+=int((v>=0.8).sum())
        counts+=np.bincount(y.cpu().numpy(),minlength=10)
    p=counts/counts.sum()
    entropy=-float(np.sum(p[p>0]*np.log(p[p>0])))/math.log(10)
    return {'prior_conf_ge08':confident/n,'prior_class_entropy':entropy,'prior_class_counts':counts.tolist()}

metrics={}
for cfg in EXPERIMENTS:
    name=cfg['name'];m=models[name].to(device)
    seed_all(31337)
    result=evaluate_recon(m,val_loader)
    result.update(eval_prior(m))
    metrics[name]=result
    (MODEL_DIR/name/'validation_metrics.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    print(name,'val MSE=',round(result['mean_mse'],6),'recon class acc=',result.get('reconstruction_class_accuracy','unavailable'))
    m.cpu()
    if torch.cuda.is_available():torch.cuda.empty_cache()

csv_write(TABLE_DIR/'validation_summary.csv',[
    dict(model=k,mean_mse=v['mean_mse'],sample_mse=v['sample_mse'],total_kl=v['total_kl'],
         pca_k90=v['pca_k90'],class_consistency=v.get('reconstruction_class_accuracy'),
         conf_ge08=v.get('prior_conf_ge08'),prior_entropy=v.get('prior_class_entropy')) for k,v in metrics.items()])
csv_write(TABLE_DIR/'class_metrics.csv',[
    dict(model=name,digit=c,mse=met['per_digit_mse'][str(c)],
         consistency=met.get('per_digit_reconstruction_accuracy',{}).get(str(c)))
    for name,met in metrics.items() for c in range(10)])
csv_write(TABLE_DIR/'kl_per_dim.csv',[
    dict(model=name,dim=j+1,kl=val,variance=metrics[name]['mean_var_by_dim'][j],
         pca_ratio=metrics[name]['pca_variance_ratio'][j])
    for name in metrics for j,val in enumerate(metrics[name]['kl_by_dim'])])

constant val MSE= 0.017643 recon class acc= 0.922
linear_warmup val MSE= 0.017029 recon class acc= 0.929
cyclical val MSE= 0.017365 recon class acc= 0.9268


## 9. 可视化：关键数字4/7/8、类别混淆、相同噪声先验生成和八维KL

In [10]:
def example_by_digit(ds):
    found={}
    for i in range(len(ds)):
        x,y=ds[i];c=int(y)
        if c not in found:found[c]=x
        if len(found)==10:break
    return sorted(found),torch.stack([found[k] for k in sorted(found)])

def show_grid(images,filepath,title,cols=8):
    n=len(images);rows=math.ceil(n/cols)
    fig,ax=plt.subplots(rows,cols,figsize=(cols*1.45,rows*1.45),squeeze=False)
    for k,a in enumerate(ax.flat):
        a.axis('off')
        if k<n:a.imshow(images[k,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
    fig.suptitle(title,fontsize=12);fig.tight_layout();fig.savefig(filepath,dpi=175);plt.close(fig)

labels,example_x=example_by_digit(val_set)
fig,ax=plt.subplots(len(labels),len(EXPERIMENTS)+1,figsize=(2.0*(len(EXPERIMENTS)+1),2.0*len(labels)),squeeze=False)
for i,c in enumerate(labels):
    ax[i,0].imshow(example_x[i,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
    ax[i,0].set_title(f'Original {c}',fontsize=8);ax[i,0].axis('off')
for j,cfg in enumerate(EXPERIMENTS,1):
    m=models[cfg['name']].to(device).eval()
    with torch.no_grad():
        mu,_=m.encode(example_x.to(device));rec=m.decode(mu).cpu()
    for i,c in enumerate(labels):
        ax[i,j].imshow(rec[i,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        ax[i,j].set_title(cfg['name'],fontsize=8);ax[i,j].axis('off')
    m.cpu()
fig.tight_layout();fig.savefig(FIG_DIR/'same_images_reconstruction.png',dpi=175);plt.close(fig)

fig,ax=plt.subplots(len(EXPERIMENTS),8,figsize=(14,2.2*len(EXPERIMENTS)),squeeze=False)
for i,cfg in enumerate(EXPERIMENTS):
    m=models[cfg['name']].to(device)
    ims=prior_images(m,8,seed=9288)
    for j in range(8):
        ax[i,j].imshow(ims[j,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest');ax[i,j].axis('off')
    ax[i,0].set_title(cfg['name']+' | same 8D noise',fontsize=8)
    show_grid(prior_images(m,PRIOR_SHOW_N,seed=7181),FIG_DIR/f'prior_{cfg["name"]}_64.png',cfg['name']+' | 8D Gaussian prior',cols=8)
    m.cpu()
fig.tight_layout();fig.savefig(FIG_DIR/'same_noise_priors.png',dpi=170);plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(14,4.2))
for i,cfg in enumerate(EXPERIMENTS):
    name=cfg['name'];y=[metrics[name]['per_digit_mse'][str(c)] for c in range(10)]
    axs[0].plot(range(10),y,marker='o',label=name)
    if PROXY_VALID:
        ys=[metrics[name]['per_digit_reconstruction_accuracy'][str(c)] for c in range(10)]
        axs[1].plot(range(10),ys,marker='o',label=name)
axs[0].set_ylabel('Validation MSE');axs[1].set_ylabel('Frozen classifier consistency (proxy)')
for a in axs:
    a.set_xticks(range(10));a.set_xlabel('True digit');a.grid(alpha=.2)
    if a.get_legend_handles_labels()[0]:a.legend(fontsize=8)
fig.tight_layout();fig.savefig(FIG_DIR/'per_class_metrics.png',dpi=170);plt.close(fig)

if PROXY_VALID:
    fig,axs=plt.subplots(1,len(EXPERIMENTS),figsize=(4.8*len(EXPERIMENTS),4.8),squeeze=False)
    for j,cfg in enumerate(EXPERIMENTS):
        name=cfg['name'];cm=np.asarray(metrics[name]['confusion'],float)
        norm=cm/np.maximum(cm.sum(1,keepdims=True),1)
        axs[0,j].imshow(norm,cmap='Blues',vmin=0,vmax=1)
        axs[0,j].set_title(name);axs[0,j].set_xticks(range(10));axs[0,j].set_yticks(range(10))
        axs[0,j].set_xlabel('Reconstructed predicted class')
        for c in (4,7,8):
            axs[0,j].text(c,c,f'{norm[c,c]:.2f}',fontsize=8,color='white',ha='center',va='center')
    fig.tight_layout();fig.savefig(FIG_DIR/'class_confusion.png',dpi=180);plt.close(fig)

fig,ax=plt.subplots(figsize=(9,4.5))
xs=np.arange(LATENT_DIM);width=.75/len(EXPERIMENTS)
for j,cfg in enumerate(EXPERIMENTS):
    ax.bar(xs+(j-(len(EXPERIMENTS)-1)/2)*width,metrics[cfg['name']]['kl_by_dim'],width,label=cfg['name'])
ax.set_xticks(xs,[str(i+1) for i in xs]);ax.set_xlabel('Latent dimension');ax.set_ylabel('Mean raw KL')
ax.legend();fig.tight_layout();fig.savefig(FIG_DIR/'kl_by_dimension.png',dpi=170);plt.close(fig)

## 10. 验证集选模：重建门槛 + 生成代理指标；正式测试仅在人工确认后
不预先宣称哪种技巧更好：需要模型在验证集重建、4/7/8 一致率、先验生成质量三方面有合理平衡。
评分使用固定阈值和候选集合归一化，属于**用于作业复核的决策规则**，不是科学上唯一合理的生成质量评价标准。

In [11]:
def norm_rank(vals,large_better):
    arr=np.asarray(vals,dtype=float)
    if not np.all(np.isfinite(arr)):raise ValueError('Candidate metrics contain missing or non-finite values')
    span=arr.max()-arr.min()
    if span<1e-12:return np.ones_like(arr)*0.5
    t=(arr-arr.min())/span
    return t if large_better else 1-t

runs=list(metrics)
min_mean=min(metrics[k]['mean_mse'] for k in runs)
min_sample=min(metrics[k]['sample_mse'] for k in runs)
eligible=[k for k in runs if metrics[k]['mean_mse']<=1.10*min_mean and metrics[k]['sample_mse']<=1.10*min_sample]
gate_relaxed=False
if not eligible:
    # MSE最优的两个模型可能互相冲突，禁止在这里崩溃或假称已通过门槛。
    gate_relaxed=True
    eligible=list(runs)
    print('WARNING: No model satisfies BOTH reconstruction gates; using all candidates ONLY for human review.')

scores={}
rec1=norm_rank([metrics[k]['mean_mse'] for k in eligible],False)
rec2=norm_rank([metrics[k]['sample_mse'] for k in eligible],False)
if PROXY_VALID:
    pr1=norm_rank([metrics[k]['prior_conf_ge08'] for k in eligible],True)
    pr2=norm_rank([metrics[k]['prior_class_entropy'] for k in eligible],True)
    for i,k in enumerate(eligible):scores[k]=float(.25*(rec1[i]+rec2[i]+pr1[i]+pr2[i]))
else:
    # 不伪装为已完成的重建+生成平衡排序；只显示临时重建候选顺序
    for i,k in enumerate(eligible):scores[k]=float(.5*(rec1[i]+rec2[i]))
recommended=max(scores,key=scores.get)
ranking=[]
for k in sorted(runs,key=lambda t:scores.get(t,-100),reverse=True):
    ranking.append({'run':k,'recon_eligible':k in eligible,'score':scores.get(k),
        'validation_mse':metrics[k]['mean_mse'],'validation_sample_mse':metrics[k]['sample_mse'],
        'reconstruction_consistency':metrics[k].get('reconstruction_class_accuracy'),
        'prior_conf_ge08':metrics[k].get('prior_conf_ge08'),
        'prior_entropy':metrics[k].get('prior_class_entropy')})
csv_write(TABLE_DIR/'selection_ranking.csv',ranking)
selection={'recommended':recommended,'status':'pending_visual_review' if PROXY_VALID else 'classifier_unreliable',
    'proxy_valid':PROXY_VALID,'eligible':eligible,'reconstruction_gate_relaxed':gate_relaxed,'scores':scores,'policy':'both MSE <= 110% of best; 50% reconstruction + 50% proxy generation if classifier valid; human inspection required'}
(TABLE_DIR/'selection_decision.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False),encoding='utf-8')
print('Recommendation (NOT confirmed):',recommended,'; status:',selection['status'])

# 可选：人工确认后再测试。不允许分类器失效时以综合评分自称已完成生成平衡选模。
if FINAL_TEST_CONFIRMED:
    if not PROXY_VALID:raise RuntimeError('Classifier unreliable; please fix proxy before final balanced selection')
    chosen=recommended if FINAL_RUN_NAME=='auto' else FINAL_RUN_NAME
    if gate_relaxed:raise RuntimeError('No candidate passed both MSE gates. Review and revise selection criteria before testing.')
    if chosen not in eligible:raise RuntimeError('Chosen model did not pass reconstruction tolerance gate')
    chosen_model=models[chosen].to(device)
    seed_all(8781)
    final_test=evaluate_recon(chosen_model,test_loader)
    (TABLE_DIR/'final_test_once.json').write_text(json.dumps({'run':chosen,'results':final_test},ensure_ascii=False,indent=2),encoding='utf-8')
    torch.save(cpu_copy(chosen_model.state_dict()),OUTPUT_ROOT/'selected_conv_vae_z8.pt')
    chosen_model.cpu()
    selection.update(status='confirmed',confirmed_run=chosen)
    (TABLE_DIR/'selection_decision.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False),encoding='utf-8')
    print('FINAL confirmed model:',chosen,'test mean MSE=',final_test['mean_mse'])
else:
    print('未使用测试集；先查看 figures/ 下的图像，然后设置 FINAL_TEST_CONFIRMED=True 再重新运行本节及打包单元格。')

Recommendation (NOT confirmed): linear_warmup ; status: pending_visual_review
未使用测试集；先查看 figures/ 下的图像，然后设置 FINAL_TEST_CONFIRMED=True 再重新运行本节及打包单元格。


## 11. 保存运行配置、全部结果与 ZIP
不覆盖以前的 `vae_three_stage_v1`；全部模型、CSV、PNG和模型检查点可供后续实验报告引用。

In [12]:
config_info={'experiment':'VAE FengHZ training tips + optional cyclical annealing','model':'ConvVAE','latent_dim':LATENT_DIM,
 'epochs':EPOCHS,'beta_max':BETA_MAX,'grad_norm_clip':GRAD_NORM_LIMIT,
 'loss_scaling':'pixel_mean_MSE + beta * per-sample-sum_KL','validation_data_only_for_selection':True,
 'train_count':len(train_set),'val_count':len(val_set),'test_count_withheld_until_confirmation':len(test_data),
 'experiments':EXPERIMENTS,'source_article':'https://fenghz.github.io/VAE-train-tips/',
 'source_cyclical':'https://github.com/haofuml/cyclical_annealing'}
(TABLE_DIR/'run_config.json').write_text(json.dumps(config_info,indent=2,ensure_ascii=False),encoding='utf-8')
readme=(
 '本实验基于已有八维卷积VAE，单独比较固定KL、线性预热和周期退火。\n'
 '区别于文章中直接替换sum/mean的建议：本实验保留每像素MSE，防止损失尺度变化。\n'
 '关键文件：tables/validation_summary.csv、tables/selection_ranking.csv、tables/selection_decision.json、figures/。\n'
 '最终测试集只在FINAL_TEST_CONFIRMED=True的情况下评价；未确认前不输出最终权重。\n'
 '周期退火参考 Fu et al., 2019，并非 FengHZ 博客原文技巧。\n'
)
(OUTPUT_ROOT/'README.txt').write_text(readme,encoding='utf-8')
zip_path=WORK/('VAE_8D_Training_Tips_Results_SMOKE.zip' if FAST_DEV_RUN else 'VAE_8D_Training_Tips_Results.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for file in OUTPUT_ROOT.rglob('*'):
        if file.is_file():zf.write(file,arcname=str(file.relative_to(OUTPUT_ROOT.parent)))
print('All outputs:',OUTPUT_ROOT,'\nZIP:',zip_path)

All outputs: /kaggle/working/hw2_results/vae_training_tips_z8_v1 
ZIP: /kaggle/working/VAE_8D_Training_Tips_Results.zip
